# Lezione 12 - Riduzione della Storia della Chat con Agent Scratchpad

Questo notebook dimostra come gestire il contesto in conversazioni lunghe utilizzando Microsoft Agent Framework. Man mano che le conversazioni crescono, il numero di token aumenta — superando infine la finestra di contesto del modello. Affrontiamo questo problema con un **modello di sintesi del contesto** e un **agent scratchpad** per una memoria persistente.

## Cosa imparerai:
1. **Perché la gestione del contesto è importante**: Comprendere i limiti dei token e le finestre di contesto
2. **Agenti consapevoli del contesto**: Costruire agenti che gestiscono il proprio contesto di conversazione
3. **Modello di sintesi del contesto**: Utilizzare strumenti per condensare la cronologia della conversazione
4. **Agent Scratchpad**: Memoria persistente che sopravvive alla riduzione del contesto

## Prerequisiti:
- Configurazione di Azure OpenAI con variabili d'ambiente impostate
- Comprensione dei concetti base degli agenti dalle lezioni precedenti


## Configurazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Perché la Gestione del Contesto è Importante

Ogni LLM ha una **finestra di contesto** finita — il numero massimo di token che può elaborare in una singola richiesta. Man mano che una conversazione multi-turno procede:

- Il **conteggio dei token cresce linearmente** con ogni messaggio dell’utente e risposta dell’assistente.
- I **token del prompt dominano il costo** perché l’intera cronologia viene reinviata a ogni turno.
- Alla fine la conversazione **supera la finestra di contesto** e il modello o tronca o genera un errore.

### Strategie per Gestire il Contesto

| Strategia | Come Funziona | Compromesso |
|---|---|---|
| **Troncamento** | Elimina i messaggi più vecchi | Perdita del contesto iniziale |
| **Sintesi** | Condensa i messaggi più vecchi in un riassunto | Alcuni dettagli persi, ma i punti chiave mantenuti |
| **Scratchpad / Memoria Esterna** | Conserva fatti chiave fuori dalla conversazione | Richiede chiamate a strumenti, ma sopravvive a qualsiasi riduzione |

In questo notebook combiniamo la **sintesi** con uno **strumento scratchpad** in modo che l’agente possa mantenere la continuità anche quando la cronologia della conversazione è condensata.


## Creazione di un Agente Consapevole del Contesto


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulazione di una lunga conversazione

Passiamo attraverso una conversazione a più turni per vedere come si accumula il contesto. L'agente dovrebbe mantenere i dettagli chiave (preferenze, budget, date di viaggio) tra i vari turni e dimostrare continuità.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Nota come l'agente conserva il contesto dai turni precedenti — conosce il Giappone, il sushi, i templi, la fotografia, il budget di 3000$, i viaggi da soli e il periodo di aprile. In una conversazione breve questo funziona bene, ma man mano che la conversazione si allunga diventa costoso rinviare l'intera cronologia.

Continuiamo la conversazione con altri turni per vedere l'accumulo del contesto:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Modello di Sintesi del Contesto

Man mano che la conversazione si sviluppa, possiamo utilizzare uno **strumento di sintesi** per condensare il contesto accumulato in un formato compatto. L'agente richiama questo strumento per registrare le preferenze chiave in modo che, anche se i messaggi più vecchi vengono eliminati, le informazioni essenziali vengano mantenute.

Questo modello è il blocco costitutivo per una riduzione della cronologia più sofisticata:
1. L'agente identifica i fatti chiave dalla conversazione
2. Richiama lo strumento di sintesi per conservarli
3. I messaggi più vecchi possono essere rimossi in sicurezza perché il sommario cattura ciò che è importante

Di seguito definiamo uno strumento `summarize_preferences` che l'agente può richiamare per registrare una sintesi compatta di ciò che ha appreso.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Sommario

In questa lezione hai imparato come gestire il contesto nelle conversazioni con agenti a lunga durata usando Microsoft Agent Framework:

### Concetti Chiave
- **Le finestre di contesto sono finite** — ogni token nella cronologia della conversazione costa denaro e conta per il limite.
- **Gli strumenti di sintesi** permettono all'agente di condensare il contesto accumulato in riassunti compatti, riducendo l'uso di token mantenendo le informazioni essenziali.
- **I blocchi note degli agenti** forniscono una memoria esterna persistente che sopravvive a qualsiasi riduzione della conversazione.

### Cosa Hai Costruito
- Un **agente consapevole del contesto** che mantiene la continuità attraverso conversazioni multi-turno
- Uno **strumento di sintesi** (`summarize_preferences`) che registra i dettagli chiave dell'utente in formato compatto
- Una **conversazione multi-turno** che dimostra la conservazione del contesto e la gestione dei cambiamenti

### Applicazioni nel Mondo Reale
- **Bot di Assistenza Clienti**: Ricordano le preferenze durante lunghe sessioni di supporto
- **Assistenti Personali**: Tengono traccia di progetti in corso senza dover spiegare nuovamente il contesto
- **Tutor Educativi**: Mantengono il progresso dello studente attraverso molte interazioni

### Passi Successivi
- Implementare un tool di blocco note completo con persistenza basata su file
- Aggiungere la troncatura automatica della cronologia dopo la sintesi
- Combinare con database vettoriali per la ricerca semantica nella memoria
- Costruire agenti in grado di riprendere conversazioni giorni dopo con il contesto completo


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avvertenza**:  
Questo documento è stato tradotto utilizzando il servizio di traduzione automatica [Co-op Translator](https://github.com/Azure/co-op-translator). Pur impegnandoci per garantire l’accuratezza, si prega di notare che le traduzioni automatizzate possono contenere errori o inesattezze. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un traduttore umano. Non ci assumiamo alcuna responsabilità per eventuali incomprensioni o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
